In [ ]:
"""Day 26 / Session 2 - Lab Assignment 1: Chat Message Format & Prompt Rendering for Conversational LLMs."""

VALID_ROLES = ("system", "user", "assistant")

conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hola, ¿cómo estás?"},
    {"role": "assistant", "content": "¡Muy bien, gracias!"},
    {"role": "user", "content": "Great, switch to English please."},
]


def make_message(role, content):
    if role not in VALID_ROLES:
        raise ValueError(f"unknown role: {role!r}")
    return {"role": role, "content": content}


def render_prompt(messages, add_generation_prompt=False):
    text = "\n".join(f"<|{m['role']}|>\n{m['content']}<|end|>" for m in messages)
    if add_generation_prompt:
        text += "\n<|assistant|>"                         # the cue that tells the model to start replying
    return text


def validate_conversation(messages):
    if not isinstance(messages, list) or not messages:
        return False
    for m in messages:
        if not isinstance(m, dict) or "role" not in m or "content" not in m:
            return False
        if m["role"] not in VALID_ROLES or not isinstance(m["content"], str):
            return False
    roles = [m["role"] for m in messages]
    if roles.count("system") > 1 or ("system" in roles and roles[0] != "system"):
        return False                                       # at most one system message, and it must come first
    rest = roles[1:] if roles[0] == "system" else roles
    expected = "user"                                      # a chat starts with the user, then strictly alternates
    for role in rest:
        if role != expected:
            return False
        expected = "assistant" if expected == "user" else "user"
    return True


def merge_system(messages):
    systems = [m["content"] for m in messages if m["role"] == "system"]
    others = [dict(m) for m in messages if m["role"] != "system"]
    if not systems:
        return others
    return [{"role": "system", "content": "\n".join(systems)}] + others     # one system message at the front
